# Carga Incremental de Google Sheet a products.catalog

## Descripción
Este notebook lee datos desde un Google Sheet y realiza una carga incremental en la tabla `workspace.products.catalog`.

## Proceso:
1. **Lectura**: Se lee la hoja 1 del Google Sheet usando la URL de exportación CSV
2. **Transformación**: Se convierte a Spark DataFrame
3. **Carga Incremental**: Se usa MERGE para:
   - Actualizar registros existentes (basado en `product_id` + `retailer`)
   - Insertar nuevos registros
4. **Verificación**: Se validan los resultados

## Google Sheet:
- **ID**: 1HWSNVwLffxLLnYf45-2QqF8br_qzkkzupd56h3RIMnU
- **Hoja**: 1 (GID=0)
- **Tabla destino**: workspace.products.catalog

## Columnas esperadas:
- product_id
- brand
- model
- main_category
- sub_category
- retailer
- link
- is_active

In [0]:
# Configuración
GOOGLE_SHEET_ID = "1HWSNVwLffxLLnYf45-2QqF8br_qzkkzupd56h3RIMnU"
GID = "0"  # Hoja 1
TARGET_TABLE = "workspace.products.catalog"

# Construir URL para exportar como CSV
csv_url = f"https://docs.google.com/spreadsheets/d/{GOOGLE_SHEET_ID}/export?format=csv&gid={GID}"

# Leer datos del Google Sheet
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime

print(f"Leyendo datos desde Google Sheet...")
df_sheet = pd.read_csv(csv_url)
print(f"Registros leídos: {len(df_sheet)}")
print(f"Columnas: {list(df_sheet.columns)}")

# Convertir a Spark DataFrame
df_new = spark.createDataFrame(df_sheet)

# Crear vista temporal
df_new.createOrReplaceTempView("new_products")

print("\nPrimeras 5 filas del Google Sheet:")
display(df_new.limit(5))

In [0]:
%sql
-- Carga incremental usando MERGE
-- Se actualizarán registros existentes y se insertarán nuevos

MERGE INTO workspace.products.catalog AS target
USING new_products AS source
ON target.product_id = source.product_id 
   AND target.retailer = source.retailer
WHEN MATCHED THEN
  UPDATE SET
    target.brand = source.brand,
    target.model = source.model,
    target.main_category = source.main_category,
    target.sub_category = source.sub_category,
    target.link = source.link,
    target.is_active = source.is_active
WHEN NOT MATCHED THEN
  INSERT (
    product_id,
    brand,
    model,
    main_category,
    sub_category,
    retailer,
    link,
    is_active
  )
  VALUES (
    source.product_id,
    source.brand,
    source.model,
    source.main_category,
    source.sub_category,
    source.retailer,
    source.link,
    source.is_active
  );

In [0]:
%sql
-- Verificar la carga
SELECT 
  COUNT(*) as total_registros,
  COUNT(DISTINCT product_id) as productos_unicos,
  COUNT(DISTINCT retailer) as retailers,
  COUNT(DISTINCT brand) as marcas,
  SUM(CASE WHEN is_active = true THEN 1 ELSE 0 END) as productos_activos
FROM workspace.products.catalog;